# Group 30 — Day 2 Experiments (Q3, Q5, Q6)
**Human or AI? | Cornell Tech Applied Machine Learning | Google Colab**

This notebook is tailored to the five competition files supplied with the task:
- `train.csv` (10,000 labeled rows; columns `id,text,label`; balanced 5,000/5,000)
- `unlabeled.csv` (3,000 rows; columns `id,text`)
- `test.csv` (3,000 rows; columns `id,text`)
- `sample_submission.csv` (3,000 rows; columns `id,label`)
- `train_starter_2000.csv` (2,000-row subset of train; not additional data)

**Experiments**
- **Q3:** confidence-based pseudo-labeling on `unlabeled.csv`, compared with supervised-only character TF-IDF + logistic regression.
- **Q5:** an equal-probability blend of character TF-IDF LR and word TF-IDF + compact style-feature LR, compared against the character baseline.
- **Q6:** record reproducibility details, metrics, environment, and optional full-data Kaggle submission output.

Do not enter any metrics into the report until the notebook has completed and produced them. The earlier Day 1 handoff metrics are context, not results from this notebook.

## 1. Upload and validate the supplied data

Run this cell first. Upload all five CSVs when prompted. The starter file is checked only to confirm it is a subset; it is not added to the training set.

In [ ]:
from pathlib import Path
import os, json, time, hashlib, platform, warnings, re, string
import numpy as np
import pandas as pd
import scipy, sklearn
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.preprocessing import StandardScaler

SEED = 20261023
np.random.seed(SEED)
DATA_DIR = Path("/content/data")
OUT_DIR = Path("/content/day2_outputs")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

required_files = ["train.csv", "unlabeled.csv", "test.csv", "sample_submission.csv"]
all_files = required_files + ["train_starter_2000.csv"]
missing = [name for name in all_files if not (DATA_DIR / name).exists()]
if missing:
    from google.colab import files
    print("Please upload these supplied CSV files:", ", ".join(missing))
    uploaded = files.upload()
    for name, content in uploaded.items():
        (DATA_DIR / name).write_bytes(content)

train_df = pd.read_csv(DATA_DIR / "train.csv")
unlabeled_df = pd.read_csv(DATA_DIR / "unlabeled.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
starter_path = DATA_DIR / "train_starter_2000.csv"
starter_df = pd.read_csv(starter_path) if starter_path.exists() else None

print("Python", platform.python_version())
print("NumPy", np.__version__, "| pandas", pd.__version__,
      "| SciPy", scipy.__version__, "| scikit-learn", sklearn.__version__)
print("Shapes:", {"train": train_df.shape, "unlabeled": unlabeled_df.shape,
                  "test": test_df.shape, "sample_submission": sample_sub.shape})
print("Train labels:", train_df["label"].value_counts().sort_index().to_dict())

In [ ]:
# Schema/integrity checks based on the supplied competition files
assert list(train_df.columns) == ["id", "text", "label"], train_df.columns.tolist()
assert list(unlabeled_df.columns) == ["id", "text"]
assert list(test_df.columns) == ["id", "text"]
assert list(sample_sub.columns) == ["id", "label"]
assert len(train_df) == 10000 and len(unlabeled_df) == 3000 and len(test_df) == 3000
assert train_df["id"].is_unique and unlabeled_df["id"].is_unique and test_df["id"].is_unique
assert set(train_df["label"].unique()) == {0, 1}
assert train_df["label"].value_counts().to_dict() == {0: 5000, 1: 5000}
assert sample_sub["id"].tolist() == test_df["id"].tolist()
for df in [train_df, unlabeled_df, test_df]:
    df["text"] = df["text"].fillna("").astype(str)

if starter_df is not None:
    assert list(starter_df.columns) == ["id", "text", "label"]
    assert starter_df["id"].isin(train_df["id"]).all()
    print("Starter set is a subset of train.csv; it will not be treated as extra data.")

print("All supplied competition-file checks passed.")

## 2. Fixed stratified train/validation split

The assignment asks for one fixed stratified 2,000-row validation set, leaving 8,000 training rows. The Day 1 handoff refers to a saved `results/split.json`, but that file was not among the CSVs supplied here. Therefore, this notebook creates a reproducible stratified split using seed `20261023`. **Do not describe it as the exact saved Day 1 split.** If your team later obtains the original split file, replace this split with those saved IDs and rerun every comparison.

In [ ]:
y_all = train_df["label"].astype(int).to_numpy()
train_idx, val_idx = train_test_split(
    np.arange(len(train_df)), test_size=2000, random_state=SEED, stratify=y_all
)
X_train = train_df.iloc[train_idx]["text"].tolist()
y_train = y_all[train_idx]
X_val = train_df.iloc[val_idx]["text"].tolist()
y_val = y_all[val_idx]
X_unlabeled = unlabeled_df["text"].tolist()

assert len(train_idx) == 8000 and len(val_idx) == 2000
print("Train class counts:", np.bincount(y_train))
print("Validation class counts:", np.bincount(y_val))

## 3. Shared model functions and supervised baseline

The baseline matches the Day 1 core winner family: character 3–5-gram TF-IDF with logistic regression, `C=3`. The validation partition is not used during fitting.

In [ ]:
def make_char_vectorizer():
    return TfidfVectorizer(
        analyzer="char", ngram_range=(3, 5), min_df=3, max_features=80000,
        lowercase=True, sublinear_tf=True, use_idf=True, smooth_idf=True,
        norm="l2", dtype=np.float64
    )

def fit_char(texts, labels):
    vec = make_char_vectorizer()
    X = vec.fit_transform(texts)
    clf = LogisticRegression(C=3.0, max_iter=2000, solver="liblinear", random_state=SEED)
    clf.fit(X, labels)
    return vec, clf

def metric_row(name, y, prob, runtime):
    prob = np.clip(np.asarray(prob), 1e-7, 1-1e-7)
    return {"method": name, "validation_auroc": float(roc_auc_score(y, prob)),
            "validation_log_loss": float(log_loss(y, prob)),
            "fit_seconds": float(runtime)}

t0 = time.time()
char_vec, char_clf = fit_char(X_train, y_train)
char_fit_seconds = time.time() - t0
p_char_val = char_clf.predict_proba(char_vec.transform(X_val))[:, 1]
baseline_row = metric_row("Supervised character TF-IDF LR C=3", y_val, p_char_val, char_fit_seconds)
print(baseline_row)

## 4. Q3 — Learning from unlabeled data (pseudo-labeling)

**Pre-fixed procedure:** score only the 3,000 rows in `unlabeled.csv` using the supervised baseline. Keep examples with probability of AI ≤ 0.02 or ≥ 0.98, with at most 750 examples from each predicted class. Refit the same character TF-IDF + LR configuration on the original labeled training rows plus those pseudo-labeled rows.

Validation rows and labels are excluded from pseudo-label generation, selection, and fitting. The thresholds and cap are fixed before validation evaluation.

In [ ]:
LOW_THRESHOLD, HIGH_THRESHOLD, MAX_PER_CLASS = 0.02, 0.98, 750
p_unlab = char_clf.predict_proba(char_vec.transform(X_unlabeled))[:, 1]

human_candidates = np.where(p_unlab <= LOW_THRESHOLD)[0]
ai_candidates = np.where(p_unlab >= HIGH_THRESHOLD)[0]
human_candidates = human_candidates[np.argsort(p_unlab[human_candidates])][:MAX_PER_CLASS]
ai_candidates = ai_candidates[np.argsort(-p_unlab[ai_candidates])][:MAX_PER_CLASS]

pseudo_idx = np.concatenate([human_candidates, ai_candidates])
pseudo_texts = [X_unlabeled[i] for i in pseudo_idx]
pseudo_labels = np.concatenate([
    np.zeros(len(human_candidates), dtype=int),
    np.ones(len(ai_candidates), dtype=int)
])
print(f"Pseudo-labels selected: {len(pseudo_idx)} total "
      f"({len(human_candidates)} human, {len(ai_candidates)} AI)")

In [ ]:
t0 = time.time()
if len(pseudo_texts):
    q3_vec, q3_clf = fit_char(X_train + pseudo_texts, np.concatenate([y_train, pseudo_labels]))
    p_q3_val = q3_clf.predict_proba(q3_vec.transform(X_val))[:, 1]
else:
    # If no unlabeled passage meets the fixed confidence rule, Q3 equals the supervised baseline.
    p_q3_val = p_char_val.copy()
q3_fit_seconds = time.time() - t0

q3_rows = [
    baseline_row,
    metric_row("Character LR + confidence pseudo-labels", y_val, p_q3_val, q3_fit_seconds)
]
q3_results = pd.DataFrame(q3_rows)
display(q3_results)
q3_results.to_csv(OUT_DIR / "q3_comparison.csv", index=False)
np.savez_compressed(
    OUT_DIR / "q3_validation_predictions.npz",
    validation_ids=train_df.iloc[val_idx]["id"].to_numpy(),
    y_true=y_val, supervised_probability=p_char_val, pseudo_probability=p_q3_val
)

### Q3 report evidence

Use `q3_comparison.csv` for the two validation AUROCs, log losses, and fit times. Report the selected pseudo-label count and class counts printed above. Discuss that high-confidence predictions are not guaranteed to be correct: incorrect pseudo-labels can reinforce the original model's mistakes. If pseudo-labeling does not improve AUROC, report the negative result honestly.

## 5. Q5 — Tested improvement: equal probability blend

**Hypothesis:** character n-grams capture subword, spelling, and punctuation patterns; word n-grams and coarse writing-style statistics capture complementary signals. Averaging the two model probabilities may improve ranking compared with the character model alone.

The second component uses word 1–2 gram TF-IDF plus compact numeric style features (length, word/sentence counts, average word length, lexical diversity, punctuation, digit, uppercase, and newline ratios). Both components are trained on the same 8,000 training rows. The blend is fixed at 50/50; no validation-based weight search is performed.

In [ ]:
def style_features(texts):
    rows = []
    for text in texts:
        words = re.findall(r"\b\w+\b", text.lower())
        sentences = [s for s in re.split(r"[.!?]+", text) if s.strip()]
        n_chars = max(len(text), 1)
        n_words = max(len(words), 1)
        rows.append([
            len(text), len(words), len(sentences), len(text)/n_words,
            float(np.mean([len(w) for w in words])) if words else 0.0,
            len(set(words))/n_words,
            sum(ch in string.punctuation for ch in text)/n_chars,
            sum(ch in ",;:" for ch in text)/n_chars,
            sum(ch.isdigit() for ch in text)/n_chars,
            sum(ch.isupper() for ch in text)/n_chars,
            sum(ch == "\n" for ch in text)/n_chars
        ])
    return np.asarray(rows, dtype=float)

word_vec = TfidfVectorizer(
    analyzer="word", ngram_range=(1, 2), min_df=2, max_features=60000,
    lowercase=True, token_pattern=r"(?u)\b\w\w+\b",
    sublinear_tf=True, smooth_idf=True, norm="l2"
)
Xw_train = word_vec.fit_transform(X_train)
Xw_val = word_vec.transform(X_val)
style_scaler = StandardScaler()
Xs_train = style_scaler.fit_transform(style_features(X_train))
Xs_val = style_scaler.transform(style_features(X_val))
Xws_train = hstack([Xw_train, csr_matrix(0.1 * Xs_train)], format="csr")
Xws_val = hstack([Xw_val, csr_matrix(0.1 * Xs_val)], format="csr")

word_style_clf = LogisticRegression(C=1.0, max_iter=2000, solver="liblinear", random_state=SEED)
t0 = time.time()
word_style_clf.fit(Xws_train, y_train)
word_style_fit_seconds = time.time() - t0
p_word_style_val = word_style_clf.predict_proba(Xws_val)[:, 1]
p_blend_val = 0.5 * p_char_val + 0.5 * p_word_style_val

q5_rows = [
    metric_row("Baseline: character TF-IDF LR C=3", y_val, p_char_val, char_fit_seconds),
    metric_row("Improvement: 50/50 character + word/style blend", y_val, p_blend_val,
               char_fit_seconds + word_style_fit_seconds)
]
q5_results = pd.DataFrame(q5_rows)
display(q5_results)
q5_results.to_csv(OUT_DIR / "q5_comparison.csv", index=False)
np.savez_compressed(
    OUT_DIR / "q5_validation_predictions.npz",
    validation_ids=train_df.iloc[val_idx]["id"].to_numpy(),
    y_true=y_val, character_probability=p_char_val,
    word_style_probability=p_word_style_val, blend_probability=p_blend_val
)
print("AUROC change (blend - baseline):",
      float(q5_results.iloc[1]["validation_auroc"] - q5_results.iloc[0]["validation_auroc"]))

### Q5 report evidence

Copy the baseline and blend AUROC/runtime from `q5_comparison.csv`. State whether the AUROC difference supports the hypothesis. Limits: one fixed split, one feature design, and one pre-fixed 50/50 blend weight. The compact handcrafted style implementation here is not guaranteed to be identical to the POS-based word/style model in the earlier Day 1 handoff, so treat this as a fresh experiment rather than claiming exact reproduction of the earlier blend.

## 6. Optional full-data refit and Kaggle submission

This section trains the Q5 blend components on all 10,000 labeled rows and writes probabilities for the supplied `test.csv`, in the exact ID order of `sample_submission.csv`. It does not upload to Kaggle. Decide whether the blend is the final model based on your measured Q5 result; if not, use the supervised character model instead.

In [ ]:
# Refit the blend on all labeled rows.
all_text = train_df["text"].tolist()
all_y = train_df["label"].astype(int).to_numpy()
test_text = test_df["text"].tolist()

final_char_vec, final_char_clf = fit_char(all_text, all_y)
p_char_test = final_char_clf.predict_proba(final_char_vec.transform(test_text))[:, 1]

final_word_vec = TfidfVectorizer(
    analyzer="word", ngram_range=(1, 2), min_df=2, max_features=60000,
    lowercase=True, token_pattern=r"(?u)\b\w\w+\b",
    sublinear_tf=True, smooth_idf=True, norm="l2"
)
X_all_word = final_word_vec.fit_transform(all_text)
X_test_word = final_word_vec.transform(test_text)
final_style_scaler = StandardScaler()
X_all_style = final_style_scaler.fit_transform(style_features(all_text))
X_test_style = final_style_scaler.transform(style_features(test_text))
X_all_combined = hstack([X_all_word, csr_matrix(0.1 * X_all_style)], format="csr")
X_test_combined = hstack([X_test_word, csr_matrix(0.1 * X_test_style)], format="csr")
final_word_clf = LogisticRegression(C=1.0, max_iter=2000, solver="liblinear", random_state=SEED)
final_word_clf.fit(X_all_combined, all_y)
p_word_test = final_word_clf.predict_proba(X_test_combined)[:, 1]

p_test_blend = 0.5 * p_char_test + 0.5 * p_word_test
submission = sample_sub[["id"]].copy()
pred_map = dict(zip(test_df["id"], p_test_blend))
submission["label"] = submission["id"].map(pred_map)
assert submission["label"].notna().all()
assert submission["id"].tolist() == test_df["id"].tolist()
assert submission["label"].between(0, 1).all()
submission.to_csv(OUT_DIR / "submission_day2_blend.csv", index=False)
print("Saved submission:", submission.shape, "to", OUT_DIR / "submission_day2_blend.csv")
display(submission.head())

## 7. Q6 — Reproducibility record

This records the actual split indices used in this notebook, data hashes, package versions, metrics, pseudo-label counts, and output names. Use it when writing Q6.1. Add real PR/commit URLs and your actual AI-use disclosure manually.

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

run_record = {
    "team": "Group Number 30",
    "seed": SEED,
    "split": {
        "method": "train_test_split(test_size=2000, stratify=labels, random_state=20261023)",
        "train_rows": int(len(train_idx)),
        "validation_rows": int(len(val_idx)),
        "train_ids": train_df.iloc[train_idx]["id"].tolist(),
        "validation_ids": train_df.iloc[val_idx]["id"].tolist()
    },
    "data_sha256": {name: sha256_file(DATA_DIR / name) for name in all_files if (DATA_DIR/name).exists()},
    "environment": {
        "python": platform.python_version(), "numpy": np.__version__,
        "pandas": pd.__version__, "scipy": scipy.__version__,
        "scikit_learn": sklearn.__version__
    },
    "q3": {
        "pseudo_label_total": int(len(pseudo_idx)),
        "pseudo_human": int(len(human_candidates)),
        "pseudo_ai": int(len(ai_candidates)),
        "thresholds": {"human_max_probability": LOW_THRESHOLD, "ai_min_probability": HIGH_THRESHOLD},
        "results": q3_rows
    },
    "q5": {"results": q5_rows, "blend_weights": {"character": 0.5, "word_style": 0.5}},
    "outputs": ["q3_comparison.csv", "q3_validation_predictions.npz",
                "q5_comparison.csv", "q5_validation_predictions.npz",
                "day2_run_record.json", "submission_day2_blend.csv"]
}
(OUT_DIR / "day2_run_record.json").write_text(json.dumps(run_record, indent=2))
print("Saved run record:", OUT_DIR / "day2_run_record.json")

## 8. Package outputs for download

Download the ZIP from the Colab Files pane (`/content/day2_outputs.zip`). It contains the tables and run record needed to write Q3/Q5 and Q6.1. For Q6.2, add real development commit links and explanations. For Q6.3, disclose tools used and what you personally checked; do not claim verification you did not perform.

In [ ]:
import shutil
zip_file = shutil.make_archive("/content/day2_outputs", "zip", OUT_DIR)
print("Created:", zip_file)